# Exploratory data analysis: CelebA-Spoof

Thin wrapper over `checkpoint1.eda`. Label statistics use every image; pixel statistics use a fixed-seed sample stratified by spoof type.

**Data root:** `CELEBA_SPOOF_ROOT` if set, else `data/CelebA_Spoof`, else the synthetic fixture in `data/fixture/CelebA_Spoof` (create it with `python -m checkpoint1.fixture`). The fixture run uses 100 images and writes to `outputs/fixture/` so it never overwrites real results. `EDA_N_SAMPLE`, `EDA_OUT_DIR` and `EDA_FIG_DIR` override the defaults.

In [1]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from checkpoint1 import data, eda

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()


def _abs(p):
    p = Path(p)
    return p if p.is_absolute() else REPO / p


# data root: $CELEBA_SPOOF_ROOT, then the real dataset, then the synthetic fixture
use_fixture = False
if 'CELEBA_SPOOF_ROOT' in os.environ:
    root = _abs(os.environ['CELEBA_SPOOF_ROOT'])
elif not data.missing_parts(_abs(data.DEFAULT_ROOT)):
    root = _abs(data.DEFAULT_ROOT)
elif not data.missing_parts(_abs(data.FIXTURE_ROOT)):
    root, use_fixture = _abs(data.FIXTURE_ROOT), True
else:
    root = _abs(data.DEFAULT_ROOT)  # run_eda exits with a readable "dataset not found" message

sub = 'outputs/fixture' if use_fixture else 'outputs'
out_dir = _abs(os.environ.get('EDA_OUT_DIR', f'{sub}/eda'))
fig_dir = _abs(os.environ.get('EDA_FIG_DIR', f'{sub}/figures'))
n_sample = int(os.environ.get('EDA_N_SAMPLE', 100 if use_fixture else 10_000))
if use_fixture:
    print(f'Using synthetic fixture at {root} (real dataset not found). Figures go to {fig_dir}.')

labels, df, report = eda.run_eda(root, n_sample, data.SEED, out_dir, fig_dir)
pd.read_csv(out_dir / 'summary.csv')

expected ~625,537 images and 10,177 subjects (published); got 65 and 8


Using synthetic fixture at /home/marcus/228/checkpoint1/data/fixture/CelebA_Spoof (real dataset not found). Figures go to /home/marcus/228/checkpoint1/outputs/fixture/figures.


,group,metric,value
0,counts,test_images,16
1,counts,test_subjects,2
2,counts,test_live,6
3,counts,test_spoof,10
4,counts,train_images,49
5,counts,train_subjects,6
6,counts,train_live,19
7,counts,train_spoof,30
8,spoof_type,Live,25
9,spoof_type,Photo,4


In [2]:
def show(*idx):
    for i in idx:
        name, title, caption = report.items[i - 1]
        display(Image(filename=fig_dir / name, width=700))
        display(Markdown(f'**Figure {i}: {title}.** {caption}'))

## Class balance

In [3]:
show(1, 2, 3, 4)

<IPython.core.display.Image object>

**Figure 1: Class balance: live vs spoof.** Spoof:live ratio is 1.6:1 (40 spoof vs 25 live), so we report per-class error rates (APCER/BPCER) and use class-weighted loss or balanced sampling instead of plain accuracy.

<IPython.core.display.Image object>

**Figure 2: Class balance: attack types.** Attack types are roughly balanced (4 to 4 images each), so per-attack metrics can be compared without reweighting the attack mix.

<IPython.core.display.Image object>

**Figure 3: Class balance: capture conditions (spoof only).** Illumination and environment are labeled only for spoof images (live images are 0), so the environment expert's auxiliary target covers spoofs only; the most common illumination class holds 25% of them.

<IPython.core.display.Image object>

**Figure 4: Class balance: subjects.** Median subject has 8 images (max 9) and 0 subjects have no live image, so splits must keep each subject inside one partition and sampling should be per subject.

## Univariate analysis

In [4]:
show(5, 6, 7, 8)

<IPython.core.display.Image object>

**Figure 5: Image resolution.** Image sizes span 160-635 px wide (median 400x418), so inputs must be resized to one shape; the pixel-count vs label Spearman rho is +0.02, which shows how far raw resolution alone could leak the label.

<IPython.core.display.Image object>

**Figure 6: Univariate: image statistics by class (full image).** The largest live/spoof gap is in sharpness (KS=0.48, where 0 = identical and 1 = fully separable), so no single image-level statistic separates the classes.

<IPython.core.display.Image object>

**Figure 7: Univariate: image statistics by presentation type.** Spread differs by presentation type; Region Mask has the highest median high-frequency ratio, so attack types leave different low-level footprints and results should be reported per attack.

<IPython.core.display.Image object>

**Figure 8: Univariate: face box size and score.** The face box covers a median 20% of the frame (live 21% vs spoof 20%), leaving about 80% of pixels as background, which is what the environment expert sees.

## Bivariate analysis

In [5]:
show(9, 10, 11, 12)

<IPython.core.display.Image object>

**Figure 9: Bivariate: feature correlation matrix.** Strongest rank correlation with the label is bg_hf (rho=-0.45) and 33 feature pairs have |rho|>0.8, so those features are largely redundant.

<IPython.core.display.Image object>

**Figure 10: Bivariate: face vs background statistics.** Median face/background sharpness ratio is 1.21 for live vs 1.29 for spoof (dashed line = equal); the gap, or lack of one, shows how much complementary signal separate face and environment experts can use.

<IPython.core.display.Image object>

**Figure 11: Bivariate: attack type vs capture conditions.** Attack types are unevenly spread over capture conditions (e.g. Photo is 100% indoor), so lighting and environment are partly confounded with attack type and results should be reported per condition.

<IPython.core.display.Image object>

**Figure 12: Bivariate: face attributes vs label.** Strongest attribute association is 5_o_Clock_Shadow (r=+0.31); a large |r| flags a possible shortcut or demographic confound to audit.

## Qualitative samples

In [6]:
show(13)

<IPython.core.display.Image object>

**Figure 13: Qualitative: random samples per type (box = face region).** Random samples per presentation type with the annotated face box; compare the cues inside the box (texture) with those outside it (glare, borders, lighting) that the face and environment experts would use.

## What this means for the project

- **Imbalance** (figures 1-3): report per-class error rates (APCER/BPCER) and per attack type; use class-weighted loss or balanced sampling.
- **Subjects** (figure 4): keep each subject in one partition; the official train/test split is subject-disjoint (see `subjects_in_both_splits` in the summary).
- **Resolution and box size** (figures 5, 8): resize inputs to one shape and check that raw resolution does not leak the label.
- **Face vs background** (figures 9-10): the more the two views differ between live and spoof, the stronger the case for separate face and environment experts.
- **Conditions and attributes** (figures 11-12): illumination/environment are spoof-only labels and may be confounded with attack type; large attribute correlations flag shortcut or bias risks.
- **Anomalies**: the `anomalies` and `sample` rows of the summary table feed the schema-sanity paragraph of the report.